# Rozšířený vzorek — všechna česká města (610)

`data/processed/all_czech_towns.csv` = všechny obce s `status_code` 3/4/5 (město / statutární město / hlavní město) z `cz_municipalities.csv` — RÚIAN registr, ne Wikipedie. 610 sídel, výrazně víc než původní `sample_seed.csv` (33, pořád platný jako kurátorovaný pilotní vzorek pro `01_full_sample.ipynb`).

**Rozsah tohoto běhu:** vzhledem k opakovaným výpadkům spojení na Overpass (viz `MasterOutline.md` §5) tohle bude běh na hodiny až dny, ne minuty. `download_all()` je na přesně tohle stavěný — jedno selhavší město dávku nezastaví, jen se zaloguje a pokračuje se dál. Klidně spusť a nech běžet na pozadí; stačí se čas od času vrátit, zkontrolovat `failed`, a spustit buňku znovu (přeskočí vše hotové, zkusí jen zbytek).

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd

from src.config import DATA_PROCESSED
from src.viz import set_style, save_fig
from src.streets.osm import download_graph, download_all
from src.streets.metrics import compute_metrics
from src.streets.clustering import build_feature_matrix, sweep_k, fit_kmeans, project_2d
from src.streets.viz import plot_polar_grid, plot_silhouette_scores, plot_cluster_scatter

set_style()

## Vzorek

In [ ]:
sample = pd.read_csv(DATA_PROCESSED / "all_czech_towns.csv", encoding="utf-8")
network_types = ["drive"]
len(sample)

## Stáhni vše (jediná síťová část)

Spusť sám, až budeš mít čas nechat to běžet dlouho. Vrátí seznam měst, co se zatím nepovedla — ty stačí zkusit později, spuštěním týž buňky znovu.

In [ ]:
failed = download_all(sample, network_types=network_types)
print(f"\n{len(failed)} selhalo: {failed}" if failed else "\nvšechno staženo")

## Spočítej metriky (čistě offline)

Bezpečné spouštět opakovaně mezi jednotlivými dávkami stahování výše — počítá jen to, pro co už je graf na disku, zbytek přeskočí a nahlásí.

In [ ]:
metrics_path = DATA_PROCESSED / "all_czech_towns_metrics.csv"

all_metrics = compute_metrics(sample, network_types=network_types, csv_path=metrics_path)
len(all_metrics), all_metrics.head()

## Shlukování a poster

Jakmile bude pokryta rozumná část vzorku (desítky až stovky měst — viz `MasterOutline.md` diskuze o minimálním n pro spolehlivé shlukování), zopakuj postup z `01_full_sample.ipynb` (volba k přes silhouette, `fit_kmeans`, `plot_polar_grid`) na `all_metrics`/`sample` odsud.